In [26]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from dotenv import load_dotenv
import os
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings

In [2]:
load_dotenv()
os.chdir("../")
data_path = "data"

In [3]:
def load_docs(data_path):
    loader = DirectoryLoader(
        data_path, glob="*.pdf", loader_cls=PyPDFLoader  # type: ignore
    )
    documents = loader.load()
    return documents


documents = load_docs(data_path)

In [4]:
documents[:5]

[Document(metadata={'producer': 'Adobe PDF Library 9.0', 'creator': 'Acrobat 9.3.3', 'creationdate': '2010-07-24T16:57:37-07:00', 'moddate': '2010-07-28T15:49:40-07:00', 'trapped': '/False', 'source': 'data/WhereThereIsNoDoctor.pdf', 'total_pages': 503, 'page': 0, 'page_label': 'i'}, page_content='Where There Is No Doctor 2010'),
 Document(metadata={'producer': 'Adobe PDF Library 9.0', 'creator': 'Acrobat 9.3.3', 'creationdate': '2010-07-24T16:57:37-07:00', 'moddate': '2010-07-28T15:49:40-07:00', 'trapped': '/False', 'source': 'data/WhereThereIsNoDoctor.pdf', 'total_pages': 503, 'page': 1, 'page_label': 'ii'}, page_content='Where There Is No Doctor 2010\nLibrary of Congress Cataloging-in-Publication Data\nThe Library of Congress has already cataloged the 10-digit ISBN as follows: \nWerner, David, 1934-\n    Where there is no doctor: a village health care handbook / by David Werner; \nwith Carol Thuman and Jane Maxwell-Rev. ed.\nIncludes Index.\nISBN 0-942364-15-5\n1. Medicine, Popular.

In [5]:
documents[5]

Document(metadata={'producer': 'Adobe PDF Library 9.0', 'creator': 'Acrobat 9.3.3', 'creationdate': '2010-07-24T16:57:37-07:00', 'moddate': '2010-07-28T15:49:40-07:00', 'trapped': '/False', 'source': 'data/WhereThereIsNoDoctor.pdf', 'total_pages': 503, 'page': 5, 'page_label': 'vi'}, page_content='Chapter 4\nHOW TO TAKE CARE OF A SICK PERSON . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 39\nThe Comfort of the Sick Person 39 Watching for Changes 41\nSpecial Care for a Person Who Is Very Ill 40 Signs of Dangerous Illness 42\nLiquids 40 When and How to Look for Medical Help 43\nFood 41 What to Tell the Health Worker 43\nCleanliness and Changing Position in Bed 41 Patient Report 44\nChapter 5\nHEALING WITHOUT MEDICINES . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 45\nHealing with Water 46\nWhen Water Is Better than Medicines 47\nChapter 6\nRIGHT AND WRONG USE OF MODERN MEDICINES. . . . . . . . . . . . . . . . . . . . . 

In [6]:
documents[5].metadata.keys()

dict_keys(['producer', 'creator', 'creationdate', 'moddate', 'trapped', 'source', 'total_pages', 'page', 'page_label'])

In [7]:
vars(documents[5])

{'id': None,
 'metadata': {'producer': 'Adobe PDF Library 9.0',
  'creator': 'Acrobat 9.3.3',
  'creationdate': '2010-07-24T16:57:37-07:00',
  'moddate': '2010-07-28T15:49:40-07:00',
  'trapped': '/False',
  'source': 'data/WhereThereIsNoDoctor.pdf',
  'total_pages': 503,
  'page': 5,
  'page_label': 'vi'},
 'page_content': 'Chapter 4\nHOW TO TAKE CARE OF A SICK PERSON . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 39\nThe Comfort of the Sick Person 39 Watching for Changes 41\nSpecial Care for a Person Who Is Very Ill 40 Signs of Dangerous Illness 42\nLiquids 40 When and How to Look for Medical Help 43\nFood 41 What to Tell the Health Worker 43\nCleanliness and Changing Position in Bed 41 Patient Report 44\nChapter 5\nHEALING WITHOUT MEDICINES . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . 45\nHealing with Water 46\nWhen Water Is Better than Medicines 47\nChapter 6\nRIGHT AND WRONG USE OF MODERN MEDICINES. . . . . . . 

In [8]:
def filter_extracted_docs_content(docs):
    return [
        Document(
            page_content=doc.page_content,
            metadata={
                "source": doc.metadata.get("source"),
                "page": doc.metadata.get("page"),
            },
        )
        for doc in docs
    ]

In [9]:
filtered_docs = filter_extracted_docs_content(documents)

In [10]:
filtered_docs[:5]

[Document(metadata={'source': 'data/WhereThereIsNoDoctor.pdf', 'page': 0}, page_content='Where There Is No Doctor 2010'),
 Document(metadata={'source': 'data/WhereThereIsNoDoctor.pdf', 'page': 1}, page_content='Where There Is No Doctor 2010\nLibrary of Congress Cataloging-in-Publication Data\nThe Library of Congress has already cataloged the 10-digit ISBN as follows: \nWerner, David, 1934-\n    Where there is no doctor: a village health care handbook / by David Werner; \nwith Carol Thuman and Jane Maxwell-Rev. ed.\nIncludes Index.\nISBN 0-942364-15-5\n1. Medicine, Popular. 2. Rural health. I. Thuman, Carol,\n1959-. II. Maxwell, Jane, 1941-. III Title.\n[DNLM: 1. Community Health Aides-handbooks.\n2. Medicine-popular works. 3. Rural Health-handbooks.\nWA 39 W492W]\nRC81.W4813 1992 610-dc20\nDNLM/DLC\n \n  92-1539\nfor Library of Congr\ness CIP\nTHIS REVISED EDITION CAN BE IMPROVED WITH  YOUR HELP.  \nIf you are a community health worker, doctor, mother, or anyone with ideas or \nsuggest

In [11]:
len(filtered_docs)

503

In [12]:
def create_chunks(documents):
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=25)
    chunked_documents = text_splitter.split_documents(documents)
    return chunked_documents


chunked_documents = create_chunks(filtered_docs)

In [13]:
chunked_documents[:5]

[Document(metadata={'source': 'data/WhereThereIsNoDoctor.pdf', 'page': 0}, page_content='Where There Is No Doctor 2010'),
 Document(metadata={'source': 'data/WhereThereIsNoDoctor.pdf', 'page': 1}, page_content='Where There Is No Doctor 2010\nLibrary of Congress Cataloging-in-Publication Data\nThe Library of Congress has already cataloged the 10-digit ISBN as follows: \nWerner, David, 1934-\n    Where there is no doctor: a village health care handbook / by David Werner; \nwith Carol Thuman and Jane Maxwell-Rev. ed.\nIncludes Index.\nISBN 0-942364-15-5\n1. Medicine, Popular. 2. Rural health. I. Thuman, Carol,\n1959-. II. Maxwell, Jane, 1941-. III Title.\n[DNLM: 1. Community Health Aides-handbooks.'),
 Document(metadata={'source': 'data/WhereThereIsNoDoctor.pdf', 'page': 1}, page_content='2. Medicine-popular works. 3. Rural Health-handbooks.\nWA 39 W492W]\nRC81.W4813 1992 610-dc20\nDNLM/DLC\n \n  92-1539\nfor Library of Congr\ness CIP\nTHIS REVISED EDITION CAN BE IMPROVED WITH  YOUR HELP.

In [14]:
len(chunked_documents)

2519

In [15]:
def setup_hf_cache():
    base_dir = os.getcwd()
    hf_home = os.path.join(base_dir, "embedding_model")
    os.makedirs(hf_home, exist_ok=True)

    os.environ["HF_HOME"] = hf_home
    os.environ["TRANSFORMERS_CACHE"] = hf_home
    os.environ["HF_DATASETS_CACHE"] = hf_home

    return hf_home


##############
def get_embedding_model():
    return HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


hf_cache_path = setup_hf_cache()
embedding_model = get_embedding_model()

#### store embedding in chroma/fiass


In [17]:
import os
from langchain_community.vectorstores import FAISS

DB_FAISS_PATH = "vectorstore/db_faiss"

if os.path.exists(DB_FAISS_PATH):
    # load existing FAISS index
    db = FAISS.load_local(
        DB_FAISS_PATH, embedding_model, allow_dangerous_deserialization=True
    )
else:
    # create new FAISS index and save
    db = FAISS.from_documents(chunked_documents, embedding_model)
    db.save_local(DB_FAISS_PATH)

In [18]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

In [19]:
custom_prompt_template = """
use the given content to give the answer to the user question.
if answer is not in the content don't make up the answer just say i don't know answer is not in the given content
dont provide answer out of the given content
content:{content}
question:{question} 
"""


def return_custom_prompt_template():
    prompt = PromptTemplate(
        template=custom_prompt_template, input_variables=["content", "question"]
    )
    return prompt

In [20]:
from langchain_groq import ChatGroq

model = ChatGroq(model="openai/gpt-oss-120b")

In [21]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [22]:
user_question="headache solutions"

In [23]:
# First, create the retriever
retriever = db.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

# Then, search with the user's question
content = retriever.invoke(user_question)
# OR
# content = retriever.get_relevant_documents(user_question)
content

[Document(id='c8a95766-b5b6-45c7-99b9-5b54fcc736b7', metadata={'source': 'data/WhereThereIsNoDoctor.pdf', 'page': 208}, page_content='162\nHEADACHES AND MIGRAINES\nSIMPLE HEADACHE can be helped by rest \nand aspirin. It often helps to put a cloth \nsoaked in hot water on the back of the \nneck and to massage (rub) the neck \nand shoulders gently. Some other home \nremedies also seem to help.\nHeadache is common with \nany sickness that causes fever. If \nheadache is severe, check for signs \nof meningitis (p. 185).\nHeadaches that keep coming back \nmay be a sign of a chronic illness or \npoor nutrition. It is important to eat well'),
 Document(id='027d58e0-9a82-432e-8fdc-672057e37abf', metadata={'source': 'data/WhereThereIsNoDoctor.pdf', 'page': 208}, page_content='and get enough sleep. If the headaches \ndo not go away, seek medical help.\nA MIGRAINE is a severe throbbing \nheadache often on one side of the head \nonly. Migraine attacks may come often, \nor months or years apart.\nA 

In [24]:
chain= (
    {"content":retriever,"question":RunnablePassthrough()}|
    return_custom_prompt_template()|model|StrOutputParser())

In [25]:
print(chain.invoke("headache solution"))

**Headache (simple)**
- Rest and take aspirin.  
- Apply a cloth soaked in hot water to the back of the neck and gently massage the neck and shoulders.  
- Use other home‑remedies that seem helpful.  
- Eat well and get enough sleep.  
- If the headache does not go away, seek medical help.  

**Migraine (severe throbbing headache)**
1. At the first sign:  
   - Take **2 aspirin** with a cup of strong coffee or strong black tea.  
   - Lie down in a dark, quiet place and try to relax.  
2. For especially bad migraines:  
   - Take aspirin with codeine or another sedative, **or** use ergotamine with caffeine (e.g., Cafergot).  
   - Start with **2 tablets** at the first sign, then **1 tablet every 30 minutes** until the pain stops, **but do not exceed 6 tablets in 24 hours**.  

These steps are taken directly from the provided text.
